## 📑 Core Legal Assignment Case Stream
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQChHg)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentcation

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import boto3
import os

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

In [0]:
table_name = 'Core_LegalAssignemntCaseStream'

schema = (
    "BusinessDate date,"
    "CIF string,"
    "AccountNumber string,"
    "PrimaryOwnerAccountNumber string,"
    "RelationTypeQC string,"
    "Queue string,"
    "AgencyLegalAssignemnt string,"
    "AssignmentDateLegalAssignment string,"
    "PacketProcessTypeLegalAssignment string,"
    "StreamStatusLegalAssignment string,"
    "MainAssignmentAgency string,"
    "LastPaymentAmount string,"
    "LastPaymentDate string,"
    "CaseLevelStremPackage string,"
    "OutstandingBalance string,"
    "DebtAmount string,"
    "FrozenInterest string,"
    "WriteOffAmount string,"
    "BusinessUnit string,"
    "CurrentBucket string,"
    "AssignmentBucket string,"
    "HostAccountStatus string,"
    "InsertTimeStamp timestamp"	
    )

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
# env_prefix ='qa' #IL: To be Deleted!

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

In [0]:
Legal_Assignemnt_Case_Stream = spark.sql(
f'''
select distinct
     BusinessDate,
     CIF,
    `Account Number` AS AccountNumber, 
    `Primary Owner Account Number` as PrimaryOwnerAccountNumber,
    `Relation Type QC` as RelationTypeQC,
     Queue,
    `Agency «Legal Assignemnt»` AS AgencyLegalAssignemnt,
    `Assignment date «Legal Assignment»` as AssignmentDateLegalAssignment,
    `Packet process type «Legal Assignment»` as PacketProcessTypeLegalAssignment,
    `Stream Status «Legal Assignment»` as StreamStatusLegalAssignment,
    `Main Assignment Agency` as MainAssignmentAgency,
    `Last Payment Amount` as LastPaymentAmount,
    `Last Payment Date` as LastPaymentDate,
    `Case Level Strem Package` as CaseLevelStremPackage,
    `Outstanding Balance` as OutstandingBalance,
    `Debt Amount` as DebtAmount,
    `Frozen Interest` as FrozenInterest,
    `Write Off Amount` as WriteOffAmount,
    `Business Unit` as BusinessUnit,
    `Current Bucket` as CurrentBucket,
    `Assignment Bucket` as AssignmentBucket,
    `Host Account Status` as HostAccountStatus
from `{env_prefix}-cdh`.landed.repointqcab_legal_assignemnt_case_stream
where BusinessDate = {run_date!r}
'''
)
Legal_Assignemnt_Case_Stream.createOrReplaceTempView("Legal_Assignemnt_Case_Stream")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
      CAST(BusinessDate AS date) AS BusinessDate
    , CAST(CIF AS string) AS CIF
    , CAST(AccountNumber AS string) AS AccountNumber
    , CAST(PrimaryOwnerAccountNumber AS string) AS PrimaryOwnerAccountNumber
    , CAST(RelationTypeQC AS string) AS RelationTypeQC
    , CAST(Queue AS string) AS Queue
    , CAST(AgencyLegalAssignemnt AS string) AS AgencyLegalAssignemnt
    , CAST(AssignmentDateLegalAssignment AS string) AS AssignmentDateLegalAssignment
    , CAST(PacketProcessTypeLegalAssignment AS string) AS PacketProcessTypeLegalAssignment
    , CAST(StreamStatusLegalAssignment AS string) AS StreamStatusLegalAssignment
    , CAST(MainAssignmentAgency AS string) AS MainAssignmentAgency
    , CAST(LastPaymentAmount AS string) AS LastPaymentAmount
    , CAST(LastPaymentDate AS string) AS LastPaymentDate
    , CAST(CaseLevelStremPackage AS string) AS CaseLevelStremPackage
    , CAST(OutstandingBalance AS string) AS OutstandingBalance
    , CAST(DebtAmount AS string) AS DebtAmount
    , CAST(FrozenInterest AS string) AS FrozenInterest
    , CAST(WriteOffAmount AS string) AS WriteOffAmount
    , CAST(BusinessUnit AS string) AS BusinessUnit
    , CAST(CurrentBucket AS string) AS CurrentBucket
    , CAST(AssignmentBucket AS string) AS AssignmentBucket
    , CAST(HostAccountStatus AS string) AS HostAccountStatus
    , CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM Legal_Assignemnt_Case_Stream
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_LegalAssignemntCaseStream: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["AccountNumber", "CIF"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
LegalAssignmentCaseStream = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
LegalAssignmentCaseStream.createOrReplaceTempView("LegalAssignmentCaseStream")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/LegalAssignmentCaseStream/"
file_name = "LegalAssignmentCaseStream_" +run_date+ ".csv"
LegalAssignmentCaseStream.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)